# M5 demand forecasting

## Notebook purpose

This notebook contains the standalone machine-learning pipeline for daily item-store demand forecasting. It excludes exploratory analysis and the upstream raw M5 preprocessing performed in the team's main notebook.

- **Input:** `m5_processed.parquet`
- **Output:** `streamlit_forecast_output.csv`
- **Forecast horizon:** 28 days
- **Validation:** final 28 historical days, forecast recursively

Set the `M5_PROCESSED_PATH` environment variable or edit `PROCESSED_DATA_PATH` in the configuration cell when the Parquet file is stored elsewhere. Set `M5_OUTPUT_DIR` to change the output directory.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 200)

FORECAST_HORIZON = 28
SERIES_KEYS = ["item_id", "store_id"]

configured_input = os.environ.get("M5_PROCESSED_PATH")
if configured_input:
    PROCESSED_DATA_PATH = Path(configured_input)
else:
    input_candidates = [
        Path("m5_processed.parquet"),
        Path("/kaggle/working/m5_processed.parquet"),
    ]
    PROCESSED_DATA_PATH = next(
        (path for path in input_candidates if path.exists()),
        input_candidates[0],
    )

default_output_directory = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path(".")
)
OUTPUT_DIRECTORY = Path(
    os.environ.get("M5_OUTPUT_DIR", str(default_output_directory))
)

print("Input path:", PROCESSED_DATA_PATH)
print("Output directory:", OUTPUT_DIRECTORY)

## Load processed data

The processed Parquet file must already contain item-store sales history, calendar and event fields, state SNAP flags, and sell price. The checks below fail early if the standalone input does not match that schema.

In [ ]:
required_source_columns = [
    "item_id", "store_id", "date", "sales",
    "dept_id", "cat_id", "state_id", "d", "wm_yr_wk",
    "weekday", "wday", "month", "year",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI", "sell_price",
]

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {PROCESSED_DATA_PATH}. "
        "Set M5_PROCESSED_PATH or edit PROCESSED_DATA_PATH."
    )

demand_data = pd.read_parquet(
    PROCESSED_DATA_PATH,
    columns=required_source_columns,
)

missing_columns = set(required_source_columns) - set(demand_data.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

demand_data["date"] = pd.to_datetime(demand_data["date"])
demand_data = demand_data.sort_values(SERIES_KEYS + ["date"]).reset_index(drop=True)

assert not demand_data.duplicated(SERIES_KEYS + ["date"]).any(), (
    "Duplicate item-store-date rows found."
)
assert demand_data["sales"].notna().all(), "Historical sales contain missing values."
assert demand_data["sales"].ge(0).all(), "Historical sales contain negative values."

print("Dataset shape:", demand_data.shape)
print("Date range:", demand_data["date"].min(), "to", demand_data["date"].max())
print("Item-store series:", demand_data[SERIES_KEYS].drop_duplicates().shape[0])

## Forecasting data preparation

All sales lags use prior rows within the same item-store series. Rolling sales statistics use `sales.shift(1)` before rolling. The final 28 days are masked before target-derived features are created, preventing validation leakage.

In [ ]:
def grouped_rolling_stats(frame, shifted_values, window, min_periods):
    groupers = [frame[column] for column in SERIES_KEYS]
    stats = (
        shifted_values
        .groupby(groupers, sort=False, observed=True)
        .rolling(window=window, min_periods=min_periods)
        .agg(["mean", "std"])
    )
    stats.index = stats.index.droplevel(list(range(len(SERIES_KEYS))))
    return stats.reindex(frame.index)


def add_calendar_features(frame):
    frame["day_of_week"] = frame["date"].dt.dayofweek.astype("int8")
    frame["week_of_year"] = frame["date"].dt.isocalendar().week.astype("int16")
    frame["day_of_month"] = frame["date"].dt.day.astype("int8")
    frame["day_of_year"] = frame["date"].dt.dayofyear.astype("int16")
    frame["quarter"] = frame["date"].dt.quarter.astype("int8")
    frame["is_weekend"] = frame["day_of_week"].isin([5, 6]).astype("int8")
    frame["has_event_1"] = frame["event_name_1"].notna().astype("int8")
    frame["has_event_2"] = frame["event_name_2"].notna().astype("int8")
    frame["snap"] = np.select(
        [
            frame["state_id"].eq("CA"),
            frame["state_id"].eq("TX"),
            frame["state_id"].eq("WI"),
        ],
        [frame["snap_CA"], frame["snap_TX"], frame["snap_WI"]],
        default=0,
    ).astype("int8")
    return frame


def add_price_features(frame):
    price_group = frame.groupby(SERIES_KEYS, sort=False, observed=True)["sell_price"]
    frame["price_available"] = frame["sell_price"].notna().astype("int8")
    frame["price_lag_7"] = price_group.shift(7).astype("float32")

    shifted_price = price_group.shift(1)
    price_stats = grouped_rolling_stats(
        frame,
        shifted_price,
        window=28,
        min_periods=7,
    )
    frame["price_rolling_28_mean"] = price_stats["mean"].astype("float32")

    valid_lag_price = frame["price_lag_7"].replace(0, np.nan)
    valid_rolling_price = frame["price_rolling_28_mean"].replace(0, np.nan)
    frame["price_change_7"] = (
        frame["sell_price"].div(valid_lag_price).sub(1).astype("float32")
    )
    frame["price_vs_rolling_28"] = (
        frame["sell_price"].div(valid_rolling_price).sub(1).astype("float32")
    )
    frame["sell_price"] = frame["sell_price"].astype("float32")
    return frame


def add_sales_history_features(frame):
    sales_group = frame.groupby(SERIES_KEYS, sort=False, observed=True)["sales"]

    for lag in [1, 7, 14, 28]:
        frame[f"sales_lag_{lag}"] = sales_group.shift(lag).astype("float32")

    shifted_sales = sales_group.shift(1)
    for window in [7, 28]:
        stats = grouped_rolling_stats(
            frame,
            shifted_sales,
            window=window,
            min_periods=window,
        )
        frame[f"sales_rolling_{window}_mean"] = stats["mean"].astype("float32")
        frame[f"sales_rolling_{window}_std"] = stats["std"].astype("float32")

    return frame

In [ ]:
validation_end = demand_data["date"].max()
validation_start = validation_end - pd.Timedelta(days=FORECAST_HORIZON - 1)
validation_mask = demand_data["date"].ge(validation_start)

assert demand_data.loc[validation_mask, "date"].nunique() == FORECAST_HORIZON, (
    f"Expected {FORECAST_HORIZON} validation dates."
)

feature_data = demand_data.copy()
feature_data["actual_sales"] = feature_data["sales"]

# Hide all holdout targets before creating lags and rolling statistics.
feature_data.loc[validation_mask, "sales"] = np.nan

feature_data = add_calendar_features(feature_data)
feature_data = add_price_features(feature_data)
feature_data = add_sales_history_features(feature_data)

feature_data["sales"] = feature_data.pop("actual_sales")
feature_data["forecast_step"] = np.where(
    feature_data["date"].ge(validation_start),
    (feature_data["date"] - validation_start).dt.days + 1,
    0,
).astype("int8")

In [ ]:
lag_features = [
    "sales_lag_1", "sales_lag_7", "sales_lag_14", "sales_lag_28",
]
rolling_features = [
    "sales_rolling_7_mean", "sales_rolling_7_std",
    "sales_rolling_28_mean", "sales_rolling_28_std",
]
calendar_features = [
    "day_of_week", "week_of_year", "day_of_month", "day_of_year",
    "month", "quarter", "year", "is_weekend",
    "has_event_1", "has_event_2", "snap",
]
price_features = [
    "sell_price", "price_available", "price_lag_7",
    "price_rolling_28_mean", "price_change_7", "price_vs_rolling_28",
]
categorical_features = [
    "item_id", "store_id", "dept_id", "cat_id", "state_id",
    "weekday", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
]
feature_columns = (
    categorical_features + calendar_features + price_features
    + lag_features + rolling_features
)
target_column = "sales"

for column in categorical_features:
    feature_data[column] = feature_data[column].astype("category")

train_data = feature_data.loc[
    feature_data["date"].lt(validation_start)
].copy()
validation_data = feature_data.loc[
    feature_data["date"].ge(validation_start)
].copy()

# Retain the cutoff history needed for later recursive validation forecasts.
validation_history = (
    train_data
    .groupby(SERIES_KEYS, sort=False, observed=True)
    .tail(max([1, 7, 14, 28]))
    .copy()
)

assert train_data["date"].max() < validation_data["date"].min()
assert validation_data["date"].nunique() == FORECAST_HORIZON
assert validation_data.groupby(SERIES_KEYS, observed=True).size().eq(
    FORECAST_HORIZON
).all()

# Only step 1 may use lag-1 actuals; later validation actuals were masked.
assert validation_data.loc[
    validation_data["forecast_step"].eq(1), "sales_lag_1"
].notna().all()
assert validation_data.loc[
    validation_data["forecast_step"].gt(1), "sales_lag_1"
].isna().all()

print("Training dates:", train_data["date"].min(), "to", train_data["date"].max())
print("Validation dates:", validation_data["date"].min(), "to", validation_data["date"].max())
print("Training shape:", train_data.shape)
print("Validation shape:", validation_data.shape)
print("Feature count:", len(feature_columns))

# Forecasting models

All four methods use the same recursive 28-day validation protocol. After each forecast date, predictions replace the unknown validation sales in history before the next date's lag and rolling features are recomputed. WAPE is reported as a percentage.

In [ ]:
def recursive_forecast(model_name, predictor):
    history = validation_history.copy()
    forecast_days = sorted(validation_data["date"].unique())
    daily_forecasts = []

    for forecast_date in forecast_days:
        current_day = validation_data.loc[
            validation_data["date"].eq(forecast_date)
        ].copy()
        current_day["sales"] = np.nan

        feature_frame = pd.concat(
            [history, current_day],
            ignore_index=True,
        ).sort_values(SERIES_KEYS + ["date"]).reset_index(drop=True)

        feature_frame = add_sales_history_features(feature_frame)
        current_features = feature_frame.loc[
            feature_frame["date"].eq(forecast_date)
        ].copy()

        predictions = pd.Series(
            predictor(current_features),
            index=current_features.index,
            dtype="float64",
        )
        if len(predictions) != len(current_features):
            raise ValueError(f"{model_name} returned an unexpected number of predictions.")
        if not np.isfinite(predictions).all():
            raise ValueError(f"{model_name} returned non-finite predictions.")

        predictions = predictions.clip(lower=0)

        forecast_day = current_features[SERIES_KEYS + ["date"]].copy()
        forecast_day["predicted_sales"] = predictions.to_numpy()
        daily_forecasts.append(forecast_day)

        current_features["sales"] = predictions.to_numpy()
        history = (
            pd.concat([history, current_features], ignore_index=True)
            .sort_values(SERIES_KEYS + ["date"])
            .groupby(SERIES_KEYS, sort=False, observed=True)
            .tail(28)
            .reset_index(drop=True)
        )

    forecast = pd.concat(daily_forecasts, ignore_index=True)
    forecast["Model"] = model_name
    return forecast


def calculate_forecast_metrics(actual, predicted):
    errors = actual - predicted
    absolute_errors = errors.abs()
    actual_total = actual.abs().sum()

    return {
        "MAE": absolute_errors.mean(),
        "RMSE": np.sqrt(np.mean(np.square(errors))),
        "WAPE": 100 * absolute_errors.sum() / actual_total if actual_total else np.nan,
    }

In [ ]:
baseline_predictors = {
    "Zero forecast": lambda frame: pd.Series(0.0, index=frame.index),
    "Seasonal naive (lag 7)": lambda frame: frame["sales_lag_7"],
    "Trailing 7-day moving average": lambda frame: frame["sales_rolling_7_mean"],
}

validation_forecasts = {}

for baseline_name, baseline_predictor in baseline_predictors.items():
    print(f"Generating {baseline_name}...")
    validation_forecasts[baseline_name] = recursive_forecast(
        baseline_name,
        baseline_predictor,
    )

In [ ]:
X_train = train_data.loc[:, feature_columns]
y_train = train_data.loc[:, target_column]

assert isinstance(X_train, pd.DataFrame)
assert all(
    isinstance(X_train[column].dtype, pd.CategoricalDtype)
    for column in categorical_features
)

lightgbm_model = LGBMRegressor(
    objective="poisson",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

lightgbm_model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
)

print("LightGBM training complete.")

In [ ]:
def lightgbm_predictor(frame):
    X_day = frame.loc[:, feature_columns]
    assert isinstance(X_day, pd.DataFrame)
    return lightgbm_model.predict(X_day)


print("Generating recursive LightGBM forecast...")
validation_forecasts["LightGBM regression"] = recursive_forecast(
    "LightGBM regression",
    lightgbm_predictor,
)

In [ ]:
validation_actuals = (
    validation_data[SERIES_KEYS + ["date", target_column]]
    .rename(columns={target_column: "actual_sales"})
)

comparison_rows = []
evaluated_forecasts = {}

for model_name, forecast in validation_forecasts.items():
    evaluated = validation_actuals.merge(
        forecast,
        on=SERIES_KEYS + ["date"],
        how="left",
        validate="one_to_one",
    )
    assert evaluated["predicted_sales"].notna().all(), (
        f"Missing predictions for {model_name}."
    )

    metrics = calculate_forecast_metrics(
        evaluated["actual_sales"],
        evaluated["predicted_sales"],
    )
    comparison_rows.append({"Model": model_name, **metrics})
    evaluated_forecasts[model_name] = evaluated

comparison_table = (
    pd.DataFrame(comparison_rows)
    .sort_values(["MAE", "RMSE", "WAPE"])
    .reset_index(drop=True)
)

comparison_table.round({"MAE": 4, "RMSE": 4, "WAPE": 2})

# Model interpretation

The fitted LightGBM model is inspected without changing its parameters or refitting it. SHAP explanations use a representative sample from the first validation day, where all series are evaluated at the same forecast information cutoff.

In [ ]:
importance_table = pd.DataFrame({
    "feature": feature_columns,
    "gain": lightgbm_model.booster_.feature_importance(importance_type="gain"),
    "split": lightgbm_model.booster_.feature_importance(importance_type="split"),
})

importance_table["gain_pct"] = (
    100 * importance_table["gain"] / importance_table["gain"].sum()
)
importance_table = importance_table.sort_values("gain", ascending=False).reset_index(drop=True)

top_n = 20
top_importance = importance_table.head(top_n).sort_values("gain")

plt.figure(figsize=(10, 7))
plt.barh(top_importance["feature"], top_importance["gain"], color="#2f6f9f")
plt.xlabel("LightGBM gain")
plt.ylabel("Feature")
plt.title(f"Top {top_n} LightGBM features by gain")
plt.tight_layout()
plt.show()

importance_table.head(top_n)

In [ ]:
shap_sample = validation_data.loc[
    validation_data["forecast_step"].eq(1),
    feature_columns,
].copy()
shap_sample = shap_sample.sample(
    n=min(2000, len(shap_sample)),
    random_state=42,
)

assert isinstance(shap_sample, pd.DataFrame)

shap_explainer = shap.TreeExplainer(lightgbm_model)
shap_values = shap_explainer.shap_values(shap_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap_values = np.asarray(shap_values)
assert shap_values.shape == shap_sample.shape

shap_importance = pd.DataFrame({
    "feature": feature_columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
})
shap_importance = shap_importance.sort_values(
    "mean_abs_shap", ascending=False
).reset_index(drop=True)

shap_importance.head(top_n)

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    shap_sample,
    plot_type="bar",
    max_display=top_n,
    show=False,
)
plt.title(f"Top {top_n} validation drivers by mean absolute SHAP")
plt.tight_layout()
plt.show()

shap.summary_plot(
    shap_values,
    shap_sample,
    max_display=top_n,
    show=False,
)
plt.title("Validation SHAP summary")
plt.tight_layout()
plt.show()

In [ ]:
business_meanings = {
    "sales_lag_1": "Yesterday's demand captures immediate momentum and replenishment patterns.",
    "sales_lag_7": "Demand seven days ago captures weekly shopping seasonality.",
    "sales_lag_14": "Demand two weeks ago captures repeated biweekly patterns.",
    "sales_lag_28": "Demand four weeks ago captures monthly or four-week recurrence.",
    "sales_rolling_7_mean": "Recent weekly average represents the current demand level.",
    "sales_rolling_28_mean": "The four-week average smooths intermittent demand and trend.",
    "sales_rolling_7_std": "Recent volatility indicates how unstable daily demand is.",
    "sales_rolling_28_std": "Longer-run volatility distinguishes consistently noisy series.",
    "sell_price": "Current price captures price level and potential demand response.",
    "price_change_7": "The seven-day price change identifies promotions or price movements.",
    "price_vs_rolling_28": "Price relative to its recent norm identifies unusually high or low prices.",
    "price_available": "Price availability acts as an item-on-sale or assortment signal.",
    "day_of_week": "Day of week captures recurring weekday and weekend shopping behavior.",
    "is_weekend": "Weekend status captures the broad weekend demand uplift.",
    "month": "Month captures broad seasonal demand differences.",
    "week_of_year": "Week of year captures annual calendar seasonality.",
    "snap": "State SNAP participation can affect purchasing behavior and demand timing.",
    "has_event_1": "Calendar-event presence captures holiday or promotional demand shifts.",
}

interpretation_table = shap_importance.head(top_n).copy()
interpretation_table["business_meaning"] = interpretation_table["feature"].map(
    business_meanings
).fillna("Item, store, calendar, event, or price context used to segment demand.")

interpretation_table

# Streamlit forecast output

Inventory status is a demand-pressure recommendation because on-hand inventory and inbound orders are not available. High risk means forecast demand exceeds the recent 28-day mean by more than one standard deviation. Medium risk means forecast demand is above the recent mean. Low risk means it is at or below the recent mean.

In [ ]:
best_model_name = comparison_table.loc[
    comparison_table["MAE"].idxmin(), "Model"
]

best_forecast = evaluated_forecasts[best_model_name][
    SERIES_KEYS + ["date", "predicted_sales"]
].copy()

inventory_benchmark = (
    validation_history
    .groupby(SERIES_KEYS, observed=True)["sales"]
    .agg(
        recent_28_day_mean="mean",
        recent_28_day_std="std",
    )
    .reset_index()
)
inventory_benchmark["recent_28_day_std"] = (
    inventory_benchmark["recent_28_day_std"].fillna(0)
)

streamlit_forecast = best_forecast.merge(
    inventory_benchmark,
    on=SERIES_KEYS,
    how="left",
    validate="many_to_one",
)

high_risk_threshold = (
    streamlit_forecast["recent_28_day_mean"]
    + streamlit_forecast["recent_28_day_std"]
)
medium_risk_threshold = streamlit_forecast["recent_28_day_mean"]

streamlit_forecast["risk_level"] = np.select(
    [
        streamlit_forecast["predicted_sales"].gt(high_risk_threshold),
        streamlit_forecast["predicted_sales"].gt(medium_risk_threshold),
    ],
    ["High", "Medium"],
    default="Low",
)

streamlit_forecast["inventory_status"] = streamlit_forecast["risk_level"].map({
    "High": "Reorder now",
    "Medium": "Monitor closely",
    "Low": "Demand within recent range",
})

streamlit_forecast["inventory_recommendation"] = streamlit_forecast["risk_level"].map({
    "High": "Prioritize replenishment and review safety stock.",
    "Medium": "Review stock coverage before the forecast date.",
    "Low": "Maintain the current replenishment cadence.",
})

streamlit_forecast["predicted_sales"] = (
    streamlit_forecast["predicted_sales"].clip(lower=0).astype("float32")
)

output_columns = [
    "item_id",
    "store_id",
    "date",
    "predicted_sales",
    "inventory_status",
    "risk_level",
    "inventory_recommendation",
]
streamlit_forecast = (
    streamlit_forecast[output_columns]
    .sort_values(["date", "store_id", "item_id"])
    .reset_index(drop=True)
)

assert not streamlit_forecast.duplicated(SERIES_KEYS + ["date"]).any()
assert streamlit_forecast[output_columns].notna().all().all()
assert streamlit_forecast["date"].nunique() == FORECAST_HORIZON

print("Selected model:", best_model_name)
print("Output shape:", streamlit_forecast.shape)
streamlit_forecast.head()

In [ ]:
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
csv_output_path = OUTPUT_DIRECTORY / "streamlit_forecast_output.csv"

streamlit_forecast.to_csv(csv_output_path, index=False)

print(f"Saved CSV: {csv_output_path}")
print("Risk distribution:")
print(streamlit_forecast["risk_level"].value_counts())